Install packages and load model/data

In [ ]:
!pip -q install transformers datasets accelerate pandas pyarrow scipy scikit-learn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE_PATH = Path("/content/drive/MyDrive/AI-Resume-Screener")

baseline_df = pd.read_parquet(BASE_PATH / "outputs/baseline_predictions.parquet")

Mounted at /content/drive


Load trained BERT

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_PATH = str(BASE_PATH / "models/bert/final")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Generate BERT predictions

In [ ]:
def predict_bert_scores(resumes, jds, batch_size=16):
    scores = []

    for start in range(0, len(resumes), batch_size):
        batch_resumes = resumes[start:start + batch_size]
        batch_jds = jds[start:start + batch_size]

        encoded = tokenizer(
            batch_resumes,
            batch_jds,
            truncation=True,
            max_length=512,
            padding=True,
            return_tensors="pt",
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            logits = model(**encoded).logits.reshape(-1)

        scores.extend(torch.clamp(logits, 0, 1).cpu().numpy().tolist())

    return scores

baseline_df["bert_score"] = predict_bert_scores(
    baseline_df["resume"].tolist(),
    baseline_df["jd"].tolist(),
)

Regression evaluation

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_regression(true_scores, predicted_scores):
    return {
        "MAE": mean_absolute_error(true_scores, predicted_scores),
        "RMSE": mean_squared_error(true_scores, predicted_scores) ** 0.5,
        "Pearson": pearsonr(true_scores, predicted_scores).statistic,
    }

model_results = pd.DataFrame({
    "TF-IDF": evaluate_regression(
        baseline_df["score"],
        baseline_df["tfidf_score"],
    ),
    "Word2Vec": evaluate_regression(
        baseline_df["score"],
        baseline_df["word2vec_score"],
    ),
    "BERT": evaluate_regression(
        baseline_df["score"],
        baseline_df["bert_score"],
    ),
}).T

display(model_results)

,MAE,RMSE,Pearson
TF-IDF,0.255893,0.399943,0.510397
Word2Vec,0.436167,0.495750,0.347788
BERT,0.155093,0.259666,0.721936


Classification-style evaluation

In [ ]:
from sklearn.metrics import classification_report

def score_to_label(score):
    if score < 0.25:
        return "No Fit"
    if score < 0.75:
        return "Potential Fit"
    return "Good Fit"

baseline_df["bert_predicted_label"] = baseline_df["bert_score"].map(score_to_label)

print(classification_report(
    baseline_df["label"],
    baseline_df["bert_predicted_label"],
    digits=3,
))

               precision    recall  f1-score   support

     Good Fit      0.807     0.289     0.425       246
       No Fit      0.783     0.939     0.854      1221
Potential Fit      0.538     0.452     0.491       533

     accuracy                          0.729      2000
    macro avg      0.709     0.560     0.590      2000
 weighted avg      0.720     0.729     0.704      2000



Save results

In [ ]:
model_results.to_csv(BASE_PATH / "outputs/model_comparison.csv")
baseline_df.to_parquet(BASE_PATH / "outputs/final_predictions.parquet", index=False)

In [ ]:
import pandas as pd

evaluation_path = (
    BASE_PATH / "outputs/feedback_human_evaluation_template.csv"
)

if evaluation_path.exists():
    feedback_evaluation = pd.read_csv(evaluation_path)

    display(feedback_evaluation.head())

    rating_columns = [
        "correctness_1_to_5",
        "usefulness_1_to_5",
        "clarity_1_to_5",
    ]

    for column in rating_columns:
        feedback_evaluation[column] = pd.to_numeric(
            feedback_evaluation[column],
            errors="coerce"
        )

    print("Average Human Evaluation Scores:")
    display(feedback_evaluation[rating_columns].mean())

    hallucination_rate = (
        feedback_evaluation["hallucination_present_yes_no"]
        .astype(str) # Convert to string type to ensure .str accessor works
        .str.lower()
        .eq("yes")
        .mean()
    )

    print(f"Hallucination Rate: {hallucination_rate * 100:.1f}%")
else:
    print("Complete Notebook 5 and ask human reviewers to fill the evaluation file.")

,sample_id,true_label,match_score,matched_skills,missing_skills,feedback,correctness_1_to_5,usefulness_1_to_5,clarity_1_to_5,hallucination_present_yes_no,reviewer_comments
0,0,No Fit,0.033192,NaN,excel,### Overall Match\n**Overall Match:** Weak mat...,NaN,NaN,NaN,NaN,NaN
1,1,Good Fit,0.028623,NaN,NaN,**Overall Match:** Weak match. The candidate's...,NaN,NaN,NaN,NaN,NaN
2,2,No Fit,0.040153,sql,"aws, excel, python",**Overall Match:** Weak match. The candidate d...,NaN,NaN,NaN,NaN,NaN
3,3,Potential Fit,0.023669,NaN,NaN,**Overall Match:** Weak match. The candidate h...,NaN,NaN,NaN,NaN,NaN
4,4,No Fit,0.027564,NaN,excel,**Overall Match:** Weak match. The candidate's...,NaN,NaN,NaN,NaN,NaN


Average Human Evaluation Scores:


,0
correctness_1_to_5,NaN
usefulness_1_to_5,NaN
clarity_1_to_5,NaN


Hallucination Rate: 0.0%
